# Colab Dense Embedding & Indexing - LegalRAG

Notebook này hỗ trợ xây dựng **Dense Vector Index** cho **toàn bộ các chiến lược chunking** và **2 mô hình embedding** theo kế hoạch nghiên cứu khoa học:

### 1. Các chiến lược Chunking hỗ trợ:
- 🏆 **`legal_aware`**: Phân đoạn theo cấu trúc pháp lý (Chương/Điều/Khoản/Điểm - phương pháp đề xuất)
- 📜 **`clause`**: Cắt theo từng Khoản luật
- 📑 **`article`**: Cắt theo từng Điều luật
- 📝 **`paragraph`**: Cắt theo từng đoạn văn
- 📏 **`fixed_300` / `fixed_500` / `fixed_800`**: Cắt cố định theo độ dài từ

### 2. Hai mô hình Embedding:
- **MiniLM-L12-v2** (`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` - 384 dim)
- **BGE-M3** (`BAAI/bge-m3` - 1024 dim)

⚡ **Chỉ cần bấm: Runtime -> Run all (hoặc `Ctrl + F9`)** để chạy tự động.

In [ ]:
# 1. Kiểm tra GPU và môi trường PyTorch CUDA
!nvidia-smi
import torch

print("\n" + "=" * 50)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
else:
    print("CẢNH BÁO: Chưa bật GPU! Hãy vào Runtime -> Change runtime type -> T4 GPU để tăng tốc.")
print("=" * 50)

In [ ]:
# 2. Clone mã nguồn LegalRAG và chuẩn bị môi trường
import os
import sys
from pathlib import Path

target_dir = Path("/content/LegalRAG")

if not (target_dir / "src" / "legal_rag").exists():
    print("Mã nguồn chưa tồn tại. Đang clone mới...")
    !rm -rf /content/LegalRAG
    !git clone https://github.com/ngminhduc129/LegalRAG.git /content/LegalRAG
else:
    print("Cập nhật mã nguồn mới nhất...")
    !git -C /content/LegalRAG pull

%cd /content/LegalRAG

!pip install sentence-transformers underthesea rank-bm25 typer rich loguru pydantic pydantic-yaml --quiet

src_dir = str((target_dir / "src").resolve())
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

import legal_rag
print(f"\n✓ Đã kết nối mã nguồn LegalRAG v{legal_rag.__version__} thành công từ: {src_dir}")

In [ ]:
# 3. Cấu hình & Build Dense Index cho các chiến lược chunking và các models
from pathlib import Path
from legal_rag.retrieval.dense import build_dense_index

# Từ điển định nghĩa các chiến lược chunking có sẵn trong repo
AVAILABLE_STRATEGIES = {
    "legal_aware": Path("data/chunks/legal_aware_run06/labor_code_2019_legal_aware.jsonl"),
    "clause":      Path("data/chunks/clause_run06/labor_code_2019_clause.jsonl"),
    "article":     Path("data/chunks/article_run06/labor_code_2019_article.jsonl"),
    "paragraph":   Path("data/chunks/paragraph_run06/labor_code_2019_paragraph.jsonl"),
    "fixed_300":   Path("data/chunks/fixed_300_run06/labor_code_2019_fixed_300.jsonl"),
    "fixed_500":   Path("data/chunks/fixed_500_run06/labor_code_2019_fixed_500.jsonl"),
    "fixed_800":   Path("data/chunks/fixed_800_run06/labor_code_2019_fixed_800.jsonl"),
}

# Từ điển định nghĩa 2 mô hình embedding
AVAILABLE_MODELS = {
    "minilm": {
        "name": "MiniLM-L12-v2",
        "model_id": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        "batch_size": 32,
    },
    "bge_m3": {
        "name": "BGE-M3",
        "model_id": "BAAI/bge-m3",
        "batch_size": 16,
    },
}

# ==========================================================================
# CẤU HÌNH LỰA CHỌN CỦA BẠN (Tùy chỉnh tại đây):
# - Để build toàn bộ: chọn ["all"]
# - Để build tập cụ thể: ví dụ ["legal_aware", "clause", "fixed_300", "fixed_500", "fixed_800"]
# ==========================================================================
RUN_STRATEGIES = ["legal_aware", "clause", "article", "fixed_300", "fixed_500", "fixed_800"]
RUN_MODELS = ["minilm", "bge_m3"]

if "all" in RUN_STRATEGIES:
    RUN_STRATEGIES = list(AVAILABLE_STRATEGIES.keys())
if "all" in RUN_MODELS:
    RUN_MODELS = list(AVAILABLE_MODELS.keys())

output_base_dir = Path("/content/LegalRAG/data/indexes/dense")
output_base_dir.mkdir(parents=True, exist_ok=True)

built_indexes = {}

for strat_name in RUN_STRATEGIES:
    chunk_file = AVAILABLE_STRATEGIES.get(strat_name)
    if not chunk_file or not chunk_file.exists():
        print(f"⚠️ Bỏ qua '{strat_name}' vì không tìm thấy file: {chunk_file}")
        continue
        
    for m_key in RUN_MODELS:
        m_info = AVAILABLE_MODELS[m_key]
        tag = f"{strat_name}_{m_key}"
        out_path = output_base_dir / f"dense_{strat_name}_{m_key}.pkl"
        
        print("\n" + "=" * 70)
        print(f"[ĐANG BUILD] Chiến lược: '{strat_name}' | Model: {m_info['name']}")
        print(f"  Input : {chunk_file}")
        print(f"  Output: {out_path}")
        print("=" * 70)
        
        idx = build_dense_index(
            chunks_path=chunk_file,
            index_path=out_path,
            model_name=m_info["model_id"],
            batch_size=m_info["batch_size"],
            max_length=256,
        )
        built_indexes[tag] = {
            "strategy": strat_name,
            "model": m_info["name"],
            "index": idx,
            "path": out_path,
            "dim": idx.embeddings.shape[1],
            "docs": len(idx.documents)
        }
        print(f"✓ Xong: {len(idx.documents)} docs | Vector shape: {idx.embeddings.shape}")

In [ ]:
# 4. Thử nghiệm so sánh Semantic Search giữa các chiến lược
test_query = "Người lao động có quyền đơn phương chấm dứt hợp đồng lao động khi nào?"

print("=" * 80)
print(f"THỬ NGHIỆM TRUY VẤN VỚI CÂU HỎI:\n\"{test_query}\"")
print("=" * 80)

for tag, item in built_indexes.items():
    print(f"\n>>> Chiến lược: [{item['strategy'].upper()}] - Model: [{item['model']}] (Tổng {item['docs']} docs) <<<")
    results = item["index"].search(test_query, k=2)
    for rank, (doc_id, score, doc) in enumerate(results, 1):
        art = doc.get("article", "N/A")
        title = doc.get("article_title", "")
        clause = doc.get("clause", "")
        prefix = f"Điều {art}"
        if clause:
            prefix += f", Khoản {clause}"
        print(f"  Top {rank} (Cosine: {score:.4f}): {prefix} - {title}")
        snippet = doc.get("content", "").replace("\n", " ").strip()
        print(f"    Nội dung: {snippet[:120]}...")

In [ ]:
# 5. Tổng kết các Index đã tạo & Hướng dẫn đóng gói tải về
import shutil

print("=" * 75)
print("TỔNG KẾT TẤT CẢ FILE INDEX ĐÃ HOÀN THÀNH:")
print("=" * 75)

for tag, item in built_indexes.items():
    size_mb = item["path"].stat().st_size / (1024 * 1024)
    print(f"- {item['strategy']:12} | {item['model']:15} | Docs: {item['docs']:4} | Dim: {item['dim']:4} | File: {item['path'].name} ({size_mb:.2f} MB)")

# Đóng gói toàn bộ các file .pkl thành file zip để tải về 1 lần
zip_output = Path("/content/LegalRAG_Dense_Indexes")
shutil.make_archive(str(zip_output), 'zip', str(output_base_dir))
print(f"\n✓ Đã nén toàn bộ index thành: {zip_output}.zip")

print("\nĐể tải toàn bộ các file index dạng zip về máy, bỏ comment dòng dưới:")
# from google.colab import files
# files.download(f"{zip_output}.zip")